# Albania Air Quality Explorer

This notebook is the first interactive JupyterLab app for exploring air quality across selected Albanian cities.


In [2]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown


In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "albania_air_quality_merged_hourly.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing dataset: {DATA_PATH}. Run src/fetch_open_meteo_air_quality.py first."
    )

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df = df.sort_values(["city", "timestamp"]).reset_index(drop=True)

display(Markdown(f"Loaded **{len(df):,}** hourly records from **{df['city'].nunique()}** Albanian cities."))
df.head()


Loaded **163,008** hourly records from **8** Albanian cities.

,city,latitude,longitude,timestamp,pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide,...,european_aqi_ozone,european_aqi_sulphur_dioxide,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,pressure_msl,date,hour
0,Berat,40.7058,19.9522,2024-01-01 00:00:00,12.8,13.6,4.7,41.5,0.6,200.5,...,17,0,7.2,89,0.0,3.9,146,1020.1,2024-01-01,0
1,Berat,40.7058,19.9522,2024-01-01 01:00:00,10.2,12.0,3.7,39.8,0.3,172.5,...,16,0,6.7,92,0.0,4.4,145,1020.1,2024-01-01,1
2,Berat,40.7058,19.9522,2024-01-01 02:00:00,9.4,10.5,2.5,38.0,0.2,157.0,...,15,0,6.6,93,0.0,6.1,140,1019.8,2024-01-01,2
3,Berat,40.7058,19.9522,2024-01-01 03:00:00,8.5,9.7,2.3,39.0,0.2,151.0,...,16,0,8.1,88,0.0,8.4,133,1019.3,2024-01-01,3
4,Berat,40.7058,19.9522,2024-01-01 04:00:00,8.2,9.9,2.0,41.0,0.2,148.0,...,16,0,9.0,83,0.0,9.9,134,1018.7,2024-01-01,4


In [4]:
pollutants = {
    "PM2.5": "pm2_5",
    "PM10": "pm10",
    "NO2": "nitrogen_dioxide",
    "O3": "ozone",
    "SO2": "sulphur_dioxide",
    "CO": "carbon_monoxide",
    "European AQI": "european_aqi",
}

city_widget = widgets.Dropdown(
    options=sorted(df["city"].unique()),
    value="Tirane",
    description="City:",
    layout=widgets.Layout(width="280px"),
)

pollutant_widget = widgets.Dropdown(
    options=list(pollutants.keys()),
    value="PM2.5",
    description="Metric:",
    layout=widgets.Layout(width="280px"),
)

freq_widget = widgets.ToggleButtons(
    options=[("Hourly", "H"), ("Daily", "D")],
    value="D",
    description="View:",
)

smooth_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=14,
    step=1,
    description="Smooth:",
    continuous_update=False,
)

output = widgets.Output()


In [7]:
def render_dashboard(*_):
    output.clear_output(wait=True)
    city = city_widget.value
    metric_label = pollutant_widget.value
    metric = pollutants[metric_label]
    freq = freq_widget.value
    smooth = smooth_widget.value

    city_df = df[df["city"] == city][["timestamp", "date", metric]].dropna().copy()

    if freq == "D":
        plot_df = city_df.groupby("date", as_index=False)[metric].mean()
        x_col = "date"
    else:
        plot_df = city_df.rename(columns={"timestamp": "date"})[["date", metric]]
        x_col = "date"

    plot_df["smoothed"] = plot_df[metric].rolling(smooth, min_periods=1).mean()

    summary = {
        "min": round(float(plot_df[metric].min()), 2),
        "mean": round(float(plot_df[metric].mean()), 2),
        "max": round(float(plot_df[metric].max()), 2),
        "latest": round(float(plot_df[metric].iloc[-1]), 2),
    }

    fig = px.line(
        plot_df,
        x=x_col,
        y=[metric, "smoothed"],
        template="plotly_white",
        title=f"{metric_label} in {city}",
        labels={"value": metric_label, "variable": "Series"},
    )
    fig.update_layout(height=520, legend_title_text="")
    fig.update_traces(line=dict(width=2))

    with output:
        display(Markdown(
            f"**Summary for {city} - {metric_label}:** min={summary['min']}, mean={summary['mean']}, max={summary['max']}, latest={summary['latest']}"
        ))
        display(fig)

for widget in [city_widget, pollutant_widget, freq_widget, smooth_widget]:
    widget.observe(render_dashboard, names="value")

controls = widgets.HBox([city_widget, pollutant_widget, freq_widget, smooth_widget])
display(controls)
display(output)
render_dashboard()


Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.Markdown object>…

In [6]:
daily_aqi = (
    df.groupby(["city", "date"], as_index=False)["european_aqi"]
    .mean()
    .sort_values(["date", "city"])
)

latest_date = daily_aqi["date"].max()
latest_snapshot = daily_aqi[daily_aqi["date"] == latest_date].sort_values("european_aqi", ascending=False)
latest_snapshot.head(10)


,city,date,european_aqi
1697,Durres,2026-04-28,39.041667
6791,Vlore,2026-04-28,35.583333
5942,Tirane,2026-04-28,35.333333
3395,Fier,2026-04-28,35.208333
5093,Shkoder,2026-04-28,33.958333
848,Berat,2026-04-28,33.416667
2546,Elbasan,2026-04-28,32.541667
4244,Korce,2026-04-28,29.208333
